In [3]:
import os

from attr import dataclass
from dotenv import load_dotenv
load_dotenv()

from google import genai
from google.genai import types

In [4]:
client = genai.Client()

In [7]:
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Ligma",
    config=types.GenerateContentConfig(
        thinking_config=types.ThinkingConfig(thinking_budget=0) # Disables thinking
    ),
)

In [8]:
print(response.text)

Ligma balls!

This is a classic internet meme! It's a play on words, where "Ligma" sounds like "lick my," leading to the punchline. It's often used as a way to trick someone into asking "What's Ligma?" so you can deliver the funny, rude response.


In [1]:
import nltk

In [61]:
nltk.download('wordnet')
nltk.download('brown')

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\saba\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package brown to
[nltk_data]     C:\Users\saba\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\brown.zip.


True

In [26]:
from nltk.corpus import wordnet as wn
from nltk.corpus import brown
import random

In [77]:
# We create this once when the script starts.
print("Loading common words filter...")
# We use a set for very fast lookups.
common_words = set(w.lower() for w in brown.words() if len(w) >= 3 and w.isalpha())
print(f"Filter created with {len(common_words)} common words.")

Loading common words filter...
Filter created with 40067 common words.


In [5]:
syns = wn.synsets('animal')
print(syns)

[Synset('animal.n.01'), Synset('animal.s.01')]


In [12]:
syns[0]

Synset('animal.n.01')

In [20]:
syns[0].hyponyms()

[Synset('scavenger.n.03'),
 Synset('male.n.01'),
 Synset('embryo.n.02'),
 Synset('migrator.n.02'),
 Synset('invertebrate.n.01'),
 Synset('mutant.n.02'),
 Synset('omnivore.n.02'),
 Synset('herbivore.n.01'),
 Synset('game.n.04'),
 Synset('prey.n.02'),
 Synset('insectivore.n.02'),
 Synset('fictional_animal.n.01'),
 Synset('domestic_animal.n.01'),
 Synset('peeper.n.03'),
 Synset('range_animal.n.01'),
 Synset('work_animal.n.01'),
 Synset('feeder.n.06'),
 Synset('predator.n.02'),
 Synset('female.n.01'),
 Synset('molter.n.01'),
 Synset('metazoan.n.01'),
 Synset('captive.n.02'),
 Synset('racer.n.03'),
 Synset('mate.n.03'),
 Synset('hexapod.n.01'),
 Synset('thoroughbred.n.03'),
 Synset('chordate.n.01'),
 Synset('darter.n.02'),
 Synset('critter.n.01'),
 Synset('survivor.n.03'),
 Synset('poikilotherm.n.01'),
 Synset('stayer.n.01'),
 Synset('young.n.01'),
 Synset('pleurodont.n.01'),
 Synset('biped.n.01'),
 Synset('pest.n.04'),
 Synset('stunt.n.02'),
 Synset('pet.n.01'),
 Synset('zooplankton.n.01')

In [21]:
def get_words_from_category(category_synset_name, max_depth=3):
    """
    Recursively gets all specific word examples (hyponyms) from a category.
    """
    words = set()
    synset = wn.synset(category_synset_name)

    # Get hyponyms recursively up to a certain depth
    for hyponym in synset.closure(lambda s: s.hyponyms(), depth=max_depth):
        for lemma in hyponym.lemmas():
            # Clean up the word and add it to our set
            word = lemma.name().replace('_', ' ').lower()
            if word.isalpha(): # Ensure it's a clean word
                words.add(word)
    return list(words)

In [28]:
get_words_from_category('science.n.01', max_depth=7)

['bioclimatology',
 'embryology',
 'rheology',
 'orology',
 'correlation',
 'cardiology',
 'paleoanthropology',
 'entomology',
 'paleogeography',
 'morphophonemics',
 'psychology',
 'strategics',
 'electromagnetics',
 'dentistry',
 'cacogenics',
 'paleornithology',
 'rhinolaryngology',
 'etymology',
 'euthenics',
 'agronomy',
 'psycholinguistics',
 'morphology',
 'mechanics',
 'lexicostatistics',
 'psychonomics',
 'rheumatology',
 'exodontia',
 'catoptrics',
 'biotech',
 'astrometry',
 'algology',
 'spelaeology',
 'paleology',
 'cytology',
 'hypsography',
 'cryptanalysis',
 'otolaryngology',
 'diachrony',
 'neuropsychiatry',
 'psychophysiology',
 'pedology',
 'foetology',
 'civics',
 'toxicology',
 'geopolitics',
 'paletiology',
 'meteorology',
 'aeromechanics',
 'enzymology',
 'trigonometry',
 'politics',
 'physiology',
 'geriatrics',
 'syntax',
 'astrophysics',
 'cryonics',
 'magnetism',
 'hematology',
 'thermostatics',
 'holography',
 'cryptanalytics',
 'cryptography',
 'cytogenetic

In [64]:
def get_words_from_specific_depth(category_synset_name, common_words_filter, target_depth=1):
    """
    Gets words from a specific depth, BUT ONLY if they are in the common_words_filter.
    """
    words = set()
    start_synset = wn.synset(category_synset_name)
    current_level_synsets = [start_synset]

    for i in range(target_depth):
        next_level_synsets = []
        for synset in current_level_synsets:
            next_level_synsets.extend(synset.hyponyms())
        current_level_synsets = next_level_synsets
        if not current_level_synsets:
            return []

    for synset in current_level_synsets:
        for lemma in synset.lemmas():
            word = lemma.name().replace('_', ' ').lower()
            # --- This is the key change! ---
            # Only add the word if it's a common word.
            if word in common_words_filter:
                words.add(word)

    return list(words)

In [70]:
get_words_from_specific_depth('animal.n.01', common_words_filter=common_words, target_depth=8)

['drake',
 'poodle',
 'layer',
 'barker',
 'deer',
 'anaconda',
 'chat',
 'seahorse',
 'grizzly',
 'panther',
 'lotte',
 'dabbler',
 'cottonmouth',
 'tarpon',
 'bunny',
 'humans',
 'sitter',
 'skate',
 'zebra',
 'dingo',
 'rattlesnake',
 'coyote',
 'stint',
 'dipper',
 'boar',
 'torpedo',
 'gander',
 'hind',
 'mule',
 'eel',
 'cheetah',
 'pig',
 'sturgeon',
 'sable',
 'kodiak',
 'sow',
 'dragon',
 'gull',
 'ass',
 'booby',
 'ounce',
 'homer',
 'ferret',
 'cowbird',
 'man',
 'knot',
 'broody',
 'smelt',
 'puppy',
 'fisher',
 'rattler',
 'roller',
 'redhead',
 'tiger',
 'newfoundland',
 'greylag',
 'catfish',
 'razorback',
 'anchovy',
 'moloch',
 'mankind',
 'world',
 'snapper',
 'hog',
 'toy',
 'lion',
 'python',
 'coney',
 'horse',
 'tumbler',
 'wildcat',
 'humanity',
 'neanderthal',
 'cur',
 'jaguar']

In [148]:
from typing import List
from dataclasses import dataclass

CATEGORIES = ['animal.n.01', 'tool.n.01', 'music.n.01', 'vehicle.n.01',
              'emotion.n.01', 'science.n.01', 'clothing.n.01', 'location.n.01',
              'food.n.01', 'art.n.01']

@dataclass
class DifficultyConfiguration:
    name_id: str = None
    num_targets: int = 3
    num_target_categories: int = 2
    num_intersected_categories: int = 2
    num_distractor_categories: int = 6
    categories: List[str] = None
    word_depth: int = 2
    rows: int = 4
    cols: int = 4
    seed: int = None

    def __post_init__(self):
        """Validate the difficulty configuration"""
        if self.categories is None:
            self.categories = CATEGORIES.copy()
        if self.name_id is None:
            self.name_id = self.generate_name_id()

        self._validate()

    def _validate(self):
        """Validate that the configuration makes sense"""
        # Basic positive integer checks
        if self.num_targets < 1:
            raise ValueError("num_targets must be at least 1.")
        if self.num_target_categories < 0:
            raise ValueError("num_target_categories must be non-negative.")
        if self.num_intersected_categories < 0:
            raise ValueError("num_intersected_categories must be non-negative.")
        if self.num_distractor_categories < 0:
            raise ValueError("num_distractor_categories must be non-negative.")
        if self.word_depth < 1:
            raise ValueError("word_depth must be at least 1.")
        if self.rows < 1 or self.cols < 1:
            raise ValueError("rows and cols must be at least 1.")

        # Board size validation
        board_size = self.rows * self.cols
        if self.num_targets >= board_size:
            raise ValueError(f"num_targets ({self.num_targets}) must be less than board size ({board_size}).")

        # Category validation - unique categories needed
        # Intersected categories are counted once but used by both targets and distractors
        unique_categories_needed = self.num_target_categories + self.num_distractor_categories - self.num_intersected_categories
        max_available_categories = len(self.categories)

        if self.num_intersected_categories > self.num_target_categories:
            raise ValueError(f"num_intersected_categories must be <= num_target_categories.")
        if self.num_intersected_categories > self.num_distractor_categories:
            raise ValueError(f"num_intersected_categories must be <= num_distractor_categories.")
        if unique_categories_needed > max_available_categories:
            raise ValueError(f"Unique categories needed ({unique_categories_needed}) exceeds available categories ({max_available_categories}).")

    def generate_name_id(self):
        return f"diff_tgts={self.num_targets}_tcats={self.num_target_categories}_icats={self.num_intersected_categories}_dcats={self.num_distractor_categories}_depth={self.word_depth}_{self.rows}x{self.cols}_seed={self.seed}"

In [11]:
from generate_data import generate_board
from configs import EASY_CONFIG, MEDIUM_CONFIG, HARD_CONFIG
from utils import format_board

In [2]:
def print_board(board_grid):
    for row in board_grid:
        print(" | ".join(f"{word:<14}" for word in row))

In [12]:
c = f"""
You are playing a round of Codenames. You are a Codemaster. Your goal is to provide a single-word clue to help your teammate guess the target words.
The target words are: {', '.join(board_data['targets'])}.
Do NOT use any of the words on the board as your clue.

Here is the game board:
---
{format_board(board_data['board'])}
---

Based on the target words {', '.join(board_data['targets'])}, provide a single clue word.
Your response must contain ONLY the clue word inside a code block. For example: ```ClueWord```
"""

In [15]:
g = f"""
You are playing a round of Codenames. You are a Guesser. Your teammate has given you a clue and a number.
Your goal is to guess {len(board_data['targets'])} words from the board that are related to the clue.

Here is the game board:
---
{format_board(board_data['board'])}
---

The clue is: "prey"
The number is: {len(board_data['targets'])}

List exactly {len(board_data['targets'])} words from the board that you think are the targets.
Your response must contain ONLY the list of guessed words inside a code block, separated by newlines.
For example, if k=3:
```
word1
word2
word3
```
"""

In [16]:
print(g)


You are playing a round of Codenames. You are a Guesser. Your teammate has given you a clue and a number.
Your goal is to guess 3 words from the board that are related to the clue.

Here is the game board:
---
feeding        | fishing        | northland      | onslaught     
rule           | chapel         | hunt           | survival      
swim           | customhouse    | immunization   | crossing      
perpetuation   | maria          | apotheosis     | action        
---

The clue is: "prey"
The number is: 3

List exactly 3 words from the board that you think are the targets.
Your response must contain ONLY the list of guessed words inside a code block, separated by newlines.
For example, if k=3:
```
word1
word2
word3
```



In [14]:
print(c)


You are playing a round of Codenames. You are a Codemaster. Your goal is to provide a single-word clue to help your teammate guess the target words.
The target words are: fishing, hunt, swim.
Do NOT use any of the words on the board as your clue.

Here is the game board:
---
feeding        | fishing        | northland      | onslaught     
rule           | chapel         | hunt           | survival      
swim           | customhouse    | immunization   | crossing      
perpetuation   | maria          | apotheosis     | action        
---

Based on the target words fishing, hunt, swim, provide a single clue word.
Your response must contain ONLY the clue word inside a code block. For example: ```ClueWord```



In [9]:
board_data = generate_board(EASY_CONFIG)
print(f"Targets: {board_data['targets']}\n")
print("Board:")
print_board(board_data['board'])

Targets: ['fishing', 'hunt', 'swim']

Board:
feeding        | fishing        | northland      | onslaught     
rule           | chapel         | hunt           | survival      
swim           | customhouse    | immunization   | crossing      
perpetuation   | maria          | apotheosis     | action        


In [24]:
from utils import get_words_from_specific_depth

In [27]:
# We create this once when the script starts.
print("Loading common words filter...")
# We use a set for very fast lookups.
common_words = set(w.lower() for w in brown.words() if len(w) >= 3 and w.isalpha())
print(f"Filter created with {len(common_words)} common words.")

Loading common words filter...
Filter created with 40067 common words.


In [54]:
get_words_from_specific_depth('event.n.01', common_words_filter=common_words, target_depth=2)

['crash',
 'disappearance',
 'involvement',
 'trouble',
 'success',
 'contingency',
 'waste',
 'takeover',
 'union',
 'battle',
 'democratization',
 'disposition',
 'involution',
 'submission',
 'going',
 'godsend',
 'eruption',
 'revolution',
 'departure',
 'destiny',
 'assessment',
 'ending',
 'gathering',
 'derivation',
 'causing',
 'activity',
 'confederation',
 'reverse',
 'socialization',
 'find',
 'dealings',
 'stampede',
 'wonder',
 'competition',
 'conclusion',
 'windfall',
 'rejection',
 'participation',
 'assemblage',
 'case',
 'leveling',
 'avalanche',
 'integration',
 'contest',
 'implementation',
 'reversal',
 'modification',
 'stop',
 'instance',
 'leaving',
 'communicating',
 'action',
 'stay',
 'wearing',
 'vote',
 'discharge',
 'collapse',
 'getting',
 'eventuality',
 'transaction',
 'stoppage',
 'interference',
 'thing',
 'emergence',
 'gravy',
 'episode',
 'fire',
 'proclamation',
 'touch',
 'residence',
 'finish',
 'retrieval',
 'recovery',
 'treat',
 'touching',
 

In [33]:
import json


def load_dataset(file_path='data/examples.jsonl'):
    """Loads a dataset from a .jsonl file into a list of dictionaries."""
    dataset = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            # json.loads() parses a single JSON string (one line)
            data_point = json.loads(line)
            dataset.append(data_point)
    return dataset

In [34]:
data = load_dataset()

In [35]:
len(data)

333

In [32]:
print(json.dumps(data[5], indent=2, ensure_ascii=False))

{
  "board": [
    [
      "track",
      "offer",
      "careerism",
      "perception"
    ],
    [
      "feeding",
      "guidance",
      "test",
      "acrobatics"
    ],
    [
      "mapping",
      "tumbling",
      "career",
      "offense"
    ],
    [
      "intervention",
      "subbing",
      "curling",
      "drill"
    ]
  ],
  "targets": [
    "acrobatics",
    "track",
    "tumbling"
  ],
  "target_categories": [
    "sport.n.01"
  ],
  "distractor_categories": [
    "vehicle.n.01",
    "activity.n.01",
    "location.n.01",
    "food.n.01"
  ],
  "intersected_categories": [],
  "difficulty": "easy",
  "word_depth": 2
}


In [36]:
data[5]

{'board': [['battle', 'bicycle', 'cart', 'involution'],
  ['revolution', 'stampede', 'villa', 'carriage'],
  ['articulation', 'eruption', 'sander', 'accompaniment'],
  ['hothouse', 'dealing', 'contest', 'consideration']],
 'targets': ['bicycle', 'carriage', 'cart'],
 'target_categories': ['vehicle.n.01'],
 'distractor_categories': ['tool.n.01',
  'communication.n.01',
  'building.n.01',
  'event.n.01'],
 'intersected_categories': [],
 'difficulty': 'easy',
 'word_depth': 2}

In [37]:
data[-1]

{'board': [['tribulation', 'procurement', 'mix', 'mingling', 'project'],
  ['allocation', 'obligation', 'right', 'thigh', 'conclusion'],
  ['church', 'progress', 'shag', 'hurl', 'crowing'],
  ['disaster', 'screeching', 'breeding', 'intervention', 'heritage'],
  ['dissection', 'conflict', 'shipbuilding', 'haul', 'evacuation']],
 'targets': ['procurement', 'right', 'thigh', 'tribulation'],
 'target_categories': ['event.n.01',
  'activity.n.01',
  'emotion.n.01',
  'body_part.n.01'],
 'distractor_categories': ['event.n.01',
  'activity.n.01',
  'emotion.n.01',
  'clothing.n.01',
  'food.n.01',
  'animal.n.01'],
 'intersected_categories': ['event.n.01', 'activity.n.01', 'emotion.n.01'],
 'difficulty': 'hard',
 'word_depth': 4}